# Data Exploration

Explore available bundles, symbols, and data quality before running backtests.

**Version**: v1.12.0  
**Architecture**: NO WRAPPERS - Direct Zipline/pandas APIs

## Purpose

This notebook helps you:
1. List all available data bundles
2. Inspect bundle metadata (symbols, date ranges)
3. Load and visualize sample data
4. Validate data quality before backtesting
5. Understand asset class characteristics (equities, forex, crypto)

**Note**: Always run this notebook first when working with new data sources.

## Setup

In [ ]:
# Add project root to path
import sys
from pathlib import Path

project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

# Standard library
import warnings
from datetime import datetime, timedelta

# Third-party
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Zipline direct imports (v1.12.0 NO WRAPPERS)
from zipline.data.bundles import bundles, load
from zipline.utils.calendar_utils import get_calendar

# Local imports - lib modules
from lib.paths import get_project_root
from lib.bundles import list_bundles, get_bundle_symbols, load_bundle
from lib.validation import (
    validate_bundle,
    ValidationConfig,
    DataValidator
)
from lib.calendars import get_calendar_for_asset_class

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('darkgrid')
warnings.filterwarnings('ignore')

print(f"✓ Setup complete")
print(f"  Project root: {get_project_root()}")

## List Available Bundles

Display all ingested data bundles.

In [ ]:
# List all available bundles (v1.12.0: Direct Zipline API)
available_bundles = list_bundles()

print(f"Total bundles available: {len(available_bundles)}\n")

if available_bundles:
    # Group bundles by asset class (heuristic based on naming)
    equity_bundles = [b for b in available_bundles if any(x in b.lower() for x in ['equity', 'spy', 'qqq', 'stock'])]
    forex_bundles = [b for b in available_bundles if any(x in b.lower() for x in ['forex', 'eur', 'usd', 'jpy', 'gbp'])]
    crypto_bundles = [b for b in available_bundles if any(x in b.lower() for x in ['crypto', 'btc', 'eth'])]
    other_bundles = [b for b in available_bundles if b not in equity_bundles + forex_bundles + crypto_bundles]
    
    if equity_bundles:
        print("Equity Bundles:")
        for bundle in sorted(equity_bundles):
            print(f"  - {bundle}")
        print()
    
    if forex_bundles:
        print("Forex Bundles:")
        for bundle in sorted(forex_bundles):
            print(f"  - {bundle}")
        print()
    
    if crypto_bundles:
        print("Crypto Bundles:")
        for bundle in sorted(crypto_bundles):
            print(f"  - {bundle}")
        print()
    
    if other_bundles:
        print("Other Bundles:")
        for bundle in sorted(other_bundles):
            print(f"  - {bundle}")
else:
    print("⚠ No bundles found. Ingest data using:")
    print("  python scripts/ingest_data.py --source yahoo --assets equities --timeframe daily")

## Select Bundle to Explore

Choose a bundle for detailed inspection.

In [ ]:
# Select bundle to explore
# Change this to your bundle name
bundle_name = available_bundles[0] if available_bundles else None

if bundle_name:
    print(f"Selected bundle: {bundle_name}")
else:
    print("⚠ No bundle selected. Set bundle_name variable above.")

## Bundle Metadata

Display symbols, date ranges, and basic statistics.

In [ ]:
if bundle_name:
    # Get symbols
    try:
        symbols = get_bundle_symbols(bundle_name)
        print(f"Symbols in bundle: {len(symbols)}")
        print(f"\nSymbols: {', '.join(symbols[:20])}{'...' if len(symbols) > 20 else ''}")
        
        # Load bundle data
        bundle_data = load_bundle(bundle_name)
        
        # Get date range
        equity_daily_bar_reader = bundle_data.equity_daily_bar_reader
        first_session = equity_daily_bar_reader.first_trading_day
        last_session = equity_daily_bar_reader.last_available_dt
        
        print(f"\nDate Range:")
        print(f"  First session: {first_session}")
        print(f"  Last session:  {last_session}")
        print(f"  Total days:    {(last_session - first_session).days}")
        
    except Exception as e:
        print(f"✗ Error loading bundle metadata: {e}")
        print(f"  This may indicate the bundle needs re-ingestion.")
else:
    print("⚠ No bundle selected")

## Validate Bundle

Run data quality checks on the bundle.

In [ ]:
if bundle_name:
    print(f"Validating bundle: {bundle_name}\n")
    
    try:
        validation_result = validate_bundle(bundle_name)
        
        if validation_result.is_valid:
            print("✓ Bundle validation PASSED")
        else:
            print("⚠ Bundle validation found issues:")
            print(validation_result.summary())
            
            # Show first few issues
            if validation_result.errors:
                print("\nErrors (first 5):")
                for check in validation_result.errors[:5]:
                    print(f"  - {check.message}")
            
            if validation_result.warnings:
                print("\nWarnings (first 5):")
                for check in validation_result.warnings[:5]:
                    print(f"  - {check.message}")
    
    except Exception as e:
        print(f"✗ Validation error: {e}")
else:
    print("⚠ No bundle selected")

## Load Sample Data

Load and inspect OHLCV data for a specific symbol.

In [ ]:
if bundle_name and symbols:
    # Select first symbol for exploration
    sample_symbol = symbols[0]
    print(f"Loading data for: {sample_symbol}\n")
    
    try:
        # Load bundle
        bundle_data = load_bundle(bundle_name)
        equity_daily_bar_reader = bundle_data.equity_daily_bar_reader
        
        # Get asset from symbol
        asset_finder = bundle_data.asset_finder
        asset = asset_finder.lookup_symbol(sample_symbol, as_of_date=None)
        
        # Load OHLCV data using Zipline's direct API (v1.12.0 NO WRAPPERS)
        sessions = equity_daily_bar_reader.sessions
        
        # Get data arrays
        opens = equity_daily_bar_reader.load_raw_arrays(
            ['open'], sessions[0], sessions[-1], [asset.sid]
        )[0][:, 0]
        highs = equity_daily_bar_reader.load_raw_arrays(
            ['high'], sessions[0], sessions[-1], [asset.sid]
        )[0][:, 0]
        lows = equity_daily_bar_reader.load_raw_arrays(
            ['low'], sessions[0], sessions[-1], [asset.sid]
        )[0][:, 0]
        closes = equity_daily_bar_reader.load_raw_arrays(
            ['close'], sessions[0], sessions[-1], [asset.sid]
        )[0][:, 0]
        volumes = equity_daily_bar_reader.load_raw_arrays(
            ['volume'], sessions[0], sessions[-1], [asset.sid]
        )[0][:, 0]
        
        # Create DataFrame
        df = pd.DataFrame({
            'open': opens,
            'high': highs,
            'low': lows,
            'close': closes,
            'volume': volumes
        }, index=sessions)
        
        # Remove NaN rows (sessions without data)
        df = df.dropna()
        
        print(f"Data shape: {df.shape}")
        print(f"Date range: {df.index[0]} to {df.index[-1]}")
        print(f"\nFirst 5 rows:")
        print(df.head())
        print(f"\nLast 5 rows:")
        print(df.tail())
        print(f"\nBasic statistics:")
        print(df.describe())
        
    except Exception as e:
        print(f"✗ Error loading data: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠ No bundle or symbols available")

## Visualize Price Data

Plot price and volume charts.

In [ ]:
if 'df' in locals() and not df.empty:
    # Create subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    # Price chart
    ax1.plot(df.index, df['close'], label='Close', linewidth=1.5)
    ax1.set_ylabel('Price', fontsize=12)
    ax1.set_title(f'{sample_symbol} - Price History', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Volume chart
    ax2.bar(df.index, df['volume'], alpha=0.7, label='Volume')
    ax2.set_ylabel('Volume', fontsize=12)
    ax2.set_xlabel('Date', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Returns distribution
    returns = df['close'].pct_change().dropna()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    ax1.hist(returns, bins=50, alpha=0.7, edgecolor='black')
    ax1.axvline(returns.mean(), color='red', linestyle='--', label=f'Mean: {returns.mean():.4f}')
    ax1.set_xlabel('Daily Returns', fontsize=12)
    ax1.set_ylabel('Frequency', fontsize=12)
    ax1.set_title('Returns Distribution', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Q-Q plot
    from scipy import stats
    stats.probplot(returns, dist="norm", plot=ax2)
    ax2.set_title('Q-Q Plot (Normality Check)', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("\nReturns Statistics:")
    print(f"  Mean:     {returns.mean():.4f}")
    print(f"  Std Dev:  {returns.std():.4f}")
    print(f"  Skewness: {returns.skew():.4f}")
    print(f"  Kurtosis: {returns.kurtosis():.4f}")
    print(f"  Min:      {returns.min():.4f}")
    print(f"  Max:      {returns.max():.4f}")
else:
    print("⚠ No data loaded")

## Data Quality Checks

Analyze data completeness, gaps, and anomalies.

In [ ]:
if 'df' in locals() and not df.empty:
    print("Data Quality Analysis\n" + "="*50)
    
    # Missing data
    missing_data = df.isnull().sum()
    print("\nMissing Values:")
    print(missing_data)
    
    # Check for gaps in date range
    date_gaps = pd.Series(df.index).diff().dt.days
    large_gaps = date_gaps[date_gaps > 7]  # Gaps > 1 week
    
    if len(large_gaps) > 0:
        print(f"\n⚠ Found {len(large_gaps)} date gaps > 7 days")
        print("Largest gaps:")
        for idx in large_gaps.nlargest(5).index:
            print(f"  {df.index[idx-1]} -> {df.index[idx]} ({large_gaps.iloc[idx]:.0f} days)")
    else:
        print("\n✓ No significant date gaps found")
    
    # Zero volume days
    zero_volume = (df['volume'] == 0).sum()
    print(f"\nZero volume days: {zero_volume} ({zero_volume/len(df)*100:.2f}%)")
    
    # Price consistency checks
    invalid_highs = (df['high'] < df['low']).sum()
    invalid_opens = ((df['open'] > df['high']) | (df['open'] < df['low'])).sum()
    invalid_closes = ((df['close'] > df['high']) | (df['close'] < df['low'])).sum()
    
    print("\nPrice Consistency:")
    print(f"  High < Low violations:  {invalid_highs}")
    print(f"  Open out of range:      {invalid_opens}")
    print(f"  Close out of range:     {invalid_closes}")
    
    if invalid_highs + invalid_opens + invalid_closes == 0:
        print("  ✓ All price data is consistent")
    
    # Outlier detection (using z-score)
    returns = df['close'].pct_change().dropna()
    z_scores = np.abs((returns - returns.mean()) / returns.std())
    outliers = z_scores[z_scores > 3]
    
    print(f"\nOutliers (|z-score| > 3): {len(outliers)}")
    if len(outliers) > 0:
        print("Top 5 outliers:")
        for date, z_score in outliers.nlargest(5).items():
            ret = returns[date]
            print(f"  {date}: return={ret:.2%}, z-score={z_score:.2f}")
else:
    print("⚠ No data loaded")

## Multi-Symbol Analysis (Optional)

Compare statistics across multiple symbols in the bundle.

In [ ]:
if bundle_name and symbols and len(symbols) > 1:
    # Analyze up to 10 symbols
    symbols_to_analyze = symbols[:min(10, len(symbols))]
    print(f"Analyzing {len(symbols_to_analyze)} symbols: {', '.join(symbols_to_analyze)}\n")
    
    summary_stats = []
    
    try:
        bundle_data = load_bundle(bundle_name)
        equity_daily_bar_reader = bundle_data.equity_daily_bar_reader
        asset_finder = bundle_data.asset_finder
        sessions = equity_daily_bar_reader.sessions
        
        for symbol in symbols_to_analyze:
            try:
                asset = asset_finder.lookup_symbol(symbol, as_of_date=None)
                
                # Load close prices
                closes = equity_daily_bar_reader.load_raw_arrays(
                    ['close'], sessions[0], sessions[-1], [asset.sid]
                )[0][:, 0]
                
                # Calculate metrics
                prices = pd.Series(closes, index=sessions).dropna()
                returns = prices.pct_change().dropna()
                
                summary_stats.append({
                    'Symbol': symbol,
                    'Data Points': len(prices),
                    'Mean Return': returns.mean(),
                    'Volatility': returns.std(),
                    'Sharpe (daily)': returns.mean() / returns.std() if returns.std() > 0 else 0,
                    'Min Price': prices.min(),
                    'Max Price': prices.max(),
                })
            except Exception as e:
                print(f"⚠ Could not analyze {symbol}: {e}")
        
        if summary_stats:
            summary_df = pd.DataFrame(summary_stats)
            print("\nSummary Statistics:")
            print(summary_df.to_string(index=False))
            
            # Plot volatility comparison
            fig, ax = plt.subplots(figsize=(12, 6))
            ax.bar(summary_df['Symbol'], summary_df['Volatility'], alpha=0.7)
            ax.set_xlabel('Symbol', fontsize=12)
            ax.set_ylabel('Daily Volatility', fontsize=12)
            ax.set_title('Volatility Comparison', fontsize=14, fontweight='bold')
            ax.grid(True, alpha=0.3)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
    
    except Exception as e:
        print(f"✗ Error in multi-symbol analysis: {e}")
else:
    print("⚠ Multi-symbol analysis requires at least 2 symbols in bundle")

## Trading Calendar Information

Inspect the trading calendar associated with this bundle.

In [ ]:
if bundle_name:
    try:
        # Infer asset class from bundle name
        if any(x in bundle_name.lower() for x in ['forex', 'eur', 'usd', 'jpy']):
            asset_class = 'forex'
        elif any(x in bundle_name.lower() for x in ['crypto', 'btc', 'eth']):
            asset_class = 'crypto'
        else:
            asset_class = 'equities'
        
        calendar_name = get_calendar_for_asset_class(asset_class)
        calendar = get_calendar(calendar_name)
        
        print(f"Asset Class: {asset_class}")
        print(f"Calendar: {calendar_name}")
        print(f"\nCalendar Properties:")
        print(f"  Name: {calendar.name}")
        print(f"  Timezone: {calendar.tz}")
        
        # Get recent trading sessions
        recent_sessions = calendar.sessions_in_range(
            pd.Timestamp.now() - pd.Timedelta(days=30),
            pd.Timestamp.now()
        )
        
        print(f"\nRecent sessions (last 10):")
        for session in recent_sessions[-10:]:
            print(f"  {session.strftime('%Y-%m-%d %A')}")
        
    except Exception as e:
        print(f"⚠ Could not load calendar info: {e}")
else:
    print("⚠ No bundle selected")

## Summary

Key findings from data exploration.

In [ ]:
print("="*60)
print("DATA EXPLORATION SUMMARY")
print("="*60)

if bundle_name:
    print(f"\nBundle: {bundle_name}")
    print(f"Symbols: {len(symbols) if 'symbols' in locals() else 'N/A'}")
    
    if 'df' in locals() and not df.empty:
        print(f"Sample Symbol: {sample_symbol}")
        print(f"Date Range: {df.index[0]} to {df.index[-1]}")
        print(f"Data Points: {len(df)}")
        print(f"\nData Quality:")
        print(f"  Zero volume days: {(df['volume'] == 0).sum()}")
        print(f"  Missing values: {df.isnull().sum().sum()}")
        
        returns = df['close'].pct_change().dropna()
        print(f"\nReturns:")
        print(f"  Mean daily return: {returns.mean():.4f}")
        print(f"  Daily volatility: {returns.std():.4f}")
        print(f"  Sharpe ratio (daily): {returns.mean()/returns.std() if returns.std() > 0 else 0:.3f}")
    
    print("\n✓ Data exploration complete")
    print("\nNext steps:")
    print("  1. Review validation results above")
    print("  2. Check for data quality issues")
    print("  3. Proceed to 01_backtest.ipynb for strategy testing")
else:
    print("\n⚠ No bundle selected for exploration")
    print("\nTo get started:")
    print("  1. Ingest data: python scripts/ingest_data.py")
    print("  2. Re-run this notebook")

print("="*60)